reference https://www.kaggle.com/code/godblessroman/rubert-fine-tuning-multiclf

In [ ]:
!pip install mlflow

In [1]:
# import io
import gc
import mlflow
import warnings
import numpy as np
import pandas as pd 

import os 
import time 
from sklearn.metrics import precision_score, recall_score, f1_score
from datetime import datetime


from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import DataLoader, Dataset
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report


warnings.simplefilter(action='ignore')

f:\HSE\year-project\toxic-messages-data-platform\toxic-messages-handling-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
USE_S3 = False
# local 
local_filename = 'dump_features_include_numeric_1.csv'
config_path = 'src/config_s3.json'

In [3]:
from src.services.utils import Boto3Reader

if USE_S3: 
    reader = Boto3Reader(config_path)
    df = reader.get_boto3_csv(
        object_path=local_filename
    )

    # clear memory and connection: 
    del reader
else: 
    df = pd.read_csv(local_filename, index_col=0)

In [4]:
df.head(3)

,raw_text_id,dataset_id,source_platform,is_verified,text_raw,is_toxic,toxicity_type,text_encoded_noninfrm,text_encoded_emoj_emotic,text_encoded_punct,...,is_bad_word_incl,has_pronouns,starts_with_cap,ends_with_dot,has_emotional_sym,has_repeating_letters_3plus,has_url,has_number,has_mention,has_hashtag
0,0,0,"2ch, pikabu",1,"Верблюдов-то за что? Дебилы, бл...\n",1.0,NaN,"Верблюдов-то за что? Дебилы, бл...","Верблюдов-то за что? Дебилы, бл...",Верблюдов[SPP_0]то за что[SPP_1] Дебилы[SPP_2]...,...,0,0,1,0,0,0,0,0,0,0
1,1,0,"2ch, pikabu",1,"Хохлы, это отдушина затюканого россиянина, мол...",1.0,NaN,"Хохлы, это отдушина затюканого россиянина, мол...","Хохлы, это отдушина затюканого россиянина, мол...",Хохлы[SPP_2] это отдушина затюканого россиянин...,...,0,0,1,1,0,0,0,0,0,0
2,2,0,"2ch, pikabu",1,Собаке - собачья смерть\n,1.0,NaN,Собаке - собачья смерть,Собаке - собачья смерть,Собаке [SPP_0] собачья смерть,...,0,0,1,0,0,0,0,0,0,0


In [6]:
TRAIN_COL = 'text_raw'
BIN_COL = 'is_toxic'
MULT_COL = 'toxicity_type'

In [7]:
# clean data: 
df.dropna(subset=TRAIN_COL, inplace=True)

print('before cleaning')
print('multilabel df shape', df[MULT_COL].shape)
print('binary shape', df[BIN_COL].shape)

df_bin = df.dropna(subset=BIN_COL)
df_mult = df.dropna(subset=MULT_COL)

print('after cleaning')
print('multilabel df shape', df_mult.shape)
print('binary df shape', df_bin.shape)

before cleaning
multilabel df shape (480618,)
binary shape (480618,)
after cleaning
multilabel df shape (397932, 35)
binary df shape (330914, 35)


In [8]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        encoding = self.tokenizer(
            text,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt',
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'label': torch.tensor(label, dtype=torch.long),
        }

def train_modeL(
                # dataset args: 
                df: pd.DataFrame, 
                text_column: str, 
                label_column: str, 
                sample_frac: float = None, # sampling fraction
                sample_n: int = None, # sampling number of counts 

                # mlflow args: 
                use_mlflow=True,
                exp_name='exp',
                save_path='exp',

                n_epochs=150,
                batch_size=8,
                random_state=42,
                learning_rate=1e-5
                ): 

    if use_mlflow:
        if exp_name is None: 
            raise ValueError(f'Please, set up the experiment name {exp_name}')
        
    # select smaller df for not an endless process: 
    if (sample_frac is not None) or (sample_n is not None) : 

        # equal counts of classes: 
        label_counts = df[label_column].value_counts()
        weights = df[label_column].map(lambda x: 1 / label_counts[x])

        df = df.sample(frac=sample_frac, weights=weights) if sample_frac is not None else \
            df.sample(n=sample_n, weights=weights)
        
    # check whether directory is not empty: 
    if len(os.listdir(save_path)) > 0: 
        cur_time = datetime.now().strftime(("%y-%d-%H:%M:%S").replace(':', '_'))
        new_path = f'{save_path}/{save_path}_{cur_time}'
        os.makedirs(new_path, exist_ok=True)

        save_path  = new_path

    df[text_column] = df[text_column].astype(str)
    
    # encode labels if they are strings: 
    encode_dict = None
    decode_dict = None
    
    if df[label_column].dtype == 'object' or df[label_column].dtype.name == 'category':
        # Create encoding dictionary for string labels
        unique_labels = sorted(df[label_column].unique())
        encode_dict = {label: idx for idx, label in enumerate(unique_labels)}
        decode_dict = {idx: label for label, idx in encode_dict.items()}
        
        # Apply encoding
        df[label_column] = df[label_column].map(encode_dict)
        print(f"Encoded labels: {encode_dict}")
    else:
        # Labels are already numeric
        df[label_column] = df[label_column].astype(int)
        unique_labels = sorted(df[label_column].unique())
        encode_dict = {label: label for label in unique_labels}
        decode_dict = encode_dict.copy()

    # train, val, test: 
    train_ids, val_ids = train_test_split(
    df.index, test_size=0.3, random_state=random_state
    )
    val_ids, test_ids = train_test_split(
        val_ids, test_size=0.4, random_state=random_state
    )

    print(f'Train shape {train_ids.shape}, val shape {val_ids.shape}, test shape {test_ids.shape}')

    train_texts = df.loc[train_ids, text_column].values
    train_labels = df.loc[train_ids, label_column].values
    val_texts = df.loc[val_ids, text_column].values
    val_labels = df.loc[val_ids, label_column].values
    # test_texts = df.loc[test_ids, text_column].values
    # test_labels = df.loc[test_ids, label_column].values
    
    tokenizer = BertTokenizer.from_pretrained('DeepPavlov/rubert-base-cased')

    train_dataset = TextDataset(train_texts, train_labels, tokenizer)
    val_dataset = TextDataset(val_texts, val_labels, tokenizer)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=True)

    model = BertForSequenceClassification.from_pretrained(
        'DeepPavlov/rubert-base-cased', 
        num_labels=len(np.unique(train_labels))
        )
    device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
    model.to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    loss_fn = torch.nn.CrossEntropyLoss()

    
    if use_mlflow: 
        mlflow.start_run()
        mlflow.log_params({
            "optimizer": optimizer.__class__.__name__,
            "learning_rate": optimizer.param_groups[0]['lr'],
            "n_epochs": n_epochs,
            "batch_size": train_loader.batch_size,
            # "scheduler": scheduler.__class__.__name__ if scheduler else "None"
        })
    
    epochs = n_epochs
    print(f"Started training for dataset shape {df.shape}")

    for epoch in range(epochs):
        st_time = time.time()
        model.train()
        train_loss = 0
        
        train_preds = []
        train_labels = []

        for batch in train_loader:
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
            # loss = outputs.loss

            loss = loss_fn(outputs.logits, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

            preds = torch.argmax(outputs.logits, dim=1)
            train_preds.extend(preds.cpu().numpy())
            train_labels.extend(labels.cpu().numpy())

        # print(f"Epoch {epoch + 1}, Training Loss: {train_loss / len(train_loader)}")

        model.eval()
        val_loss = 0

        val_preds = []
        val_labels = []
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['label'].to(device)
                outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
                # loss = outputs.loss

                loss = loss_fn(outputs.logits, labels)
                val_loss += loss.item()
                preds = torch.argmax(outputs.logits, dim=1)

                val_preds.extend(preds.cpu().numpy())
                val_labels.extend(labels.cpu().numpy())
                
        print(f"Epoch {epoch + 1}, Training Loss: {val_loss / len(val_loader)}")

        # print(f"Epoch {epoch + 1}, Validation Loss: {val_loss / len(val_loader)}")
        # print(classification_report(all_labels, all_preds))

        train_precision = precision_score(train_labels, train_preds, average='weighted', zero_division=0)
        train_recall = recall_score(train_labels, train_preds, average='weighted')
        train_f1 = f1_score(train_labels, train_preds, average='weighted')
        # print(f"Training Precision: {train_precision}, Training Recall: {train_recall}")

        val_precision = precision_score(val_labels, val_preds, average='weighted', zero_division=0)
        val_recall = recall_score(val_labels, val_preds, average='weighted')
        val_f1 = f1_score(val_labels, val_preds, average='weighted')
        # print(f"Validation Precision: {val_precision}, Validation Recall: {val_recall}")

        end_time = time.time()
        epoch_f_time = end_time - st_time

        # log weights for every epoch: 
        model.save_pretrained(f'./{save_path}/')
        if epoch==0:
            tokenizer.save_pretrained(f'./{save_path}/')
        
        if use_mlflow:
            mlflow.log_metrics({
                # "train_loss": np.round(np.mean(train_loss), 2),
                # "val_precision": np.round(train_precision, 2),
                # "val_recall": np.round(train_recall, 2),

                "train_loss": np.round(train_loss / len(train_loader), 2),
                "train_precision": np.round(train_precision, 2),
                "train_recall": np.round(train_recall, 2),
                "train_f1": np.round(train_f1, 2),

                "val_loss": np.round(val_loss / len(val_loader), 2),
                "val_precision": np.round(val_precision, 2),
                "val_recall": np.round(val_recall, 2),
                "val_f1": np.round(val_f1, 2),

                "epoch_time": np.round(epoch_f_time, 2) 
            }, 
            step=epoch)

            # mlflow.pytorch.log_model(model, 
            #                          artifact_path=f"{save_path}/epoch_{epoch}")
            # mlflow.pytorch.log_model(tokenizer, artifact_path=f"{save_path}/epoch_{epoch}")

    if use_mlflow: 
        mlflow.end_run()

    # export: 
    print(f'Model and tokenizer path: ./{save_path}/')

    # return model, tokenizer, csv test set, and encoding dicts: 
    return (model, tokenizer, df.loc[test_ids, :], {'encode': encode_dict, 'decode': decode_dict})

## Set up MLFlow params

In [ ]:
# run mlflow locally: 
!mlflow ui

In [9]:
USE_MLFLOW = True
host = "http://localhost:5000"

## Train binary/multilabel classifier

In [ ]:
gc.collect()


LR = 1e-5
N_EPOCHS = 40
N_ROWS = 10000
# N_ROWS = 100


experiment_name = f'binary-toxicity-{N_EPOCHS}_epochs-{N_ROWS}_samples'
mlflow.set_tracking_uri(host)
mlflow.set_experiment(experiment_name)


try: 
    model, tokenizer, test_df, label_mapping = train_modeL(
        # df=df_bin, # binary classification 
        df=df_mult, # multilabel classification 
        sample_n=N_ROWS,
        batch_size=8,
        learning_rate=LR,
        text_column=TRAIN_COL,
        # label_column=BIN_COL,
        label_column=MULT_COL,
        use_mlflow=USE_MLFLOW,
        exp_name=experiment_name,
        n_epochs=N_EPOCHS,
        save_path='exp_binary'
    )
    print(f"Label encoding: {label_mapping['encode']}")
    print(f"Label decoding: {label_mapping['decode']}")
except Exception as ex: 
    print(ex)
finally: 
    if USE_MLFLOW: 
        mlflow.end_run()

## Compare and test ClassicML vs Bert decisions

In [16]:
def predict_toxicity(text, model, tokenizer, device, max_len=128):
    model.eval()
    encoding = tokenizer(
        text,
        max_length=max_len,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=1)
        pred_label = torch.argmax(probs, dim=1).item()
        
    return pred_label, probs

def batch_predict(texts, model, tokenizer, device, batch_size=32):
    model.eval()
    predictions = []
    
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        
        encodings = tokenizer(
            batch_texts,
            max_length=128,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        input_ids = encodings['input_ids'].to(device)
        attention_mask = encodings['attention_mask'].to(device)
        
        with torch.no_grad():
            outputs = model(input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            preds = torch.argmax(logits, dim=1)
            predictions.extend(preds.cpu().numpy())
            
    return predictions

In [ ]:
# unzipped archive: 
model_path = './binary_files/'
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

# select test df: 
test_df = pd.read_csv('test_df.csv', index_col=0)

# select test texts from df: 
test_texts = test_df[TRAIN_COL].tolist()


### Bert prediction

In [54]:
# init clasifier, predict: 
bert_model = BertForSequenceClassification.from_pretrained(model_path)
bert_tokenizer = BertTokenizer.from_pretrained(model_path)
bert_model.to(device)

bert_predictions = batch_predict(test_texts, bert_model, bert_tokenizer, device)
test_df['bert_predictions'] = bert_predictions

# calculate and print metrics: 
print("BERT Model Performance on Test Set:")
print(classification_report(test_df[BIN_COL], test_df['bert_predictions']))

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6695.70it/s]


BERT Model Performance on Test Set:
              precision    recall  f1-score   support

           0       0.89      0.91      0.90       607
           1       0.91      0.89      0.90       593

    accuracy                           0.90      1200
   macro avg       0.90      0.90      0.90      1200
weighted avg       0.90      0.90      0.90      1200



### LinearSVC prediction

In [ ]:
import pickle
import io
import numpy as np
import scipy.sparse as sp
from src.services.utils import Boto3Reader, load_config

# config files and s3 loader: 
s3_config_path = 'src/config_s3.json'
config = load_config(s3_config_path)
predictor_config = config['predictors'][0]
reader = Boto3Reader(config_path=s3_config_path, bucket_name=predictor_config['bucket_name'])

# inderence precprocessing: 
model_bytes = reader.get_boto3_obj(predictor_config['model_path'])
sklearn_model = pickle.load(io.BytesIO(model_bytes['Body'].read()))
encoder_bytes = reader.get_boto3_obj(predictor_config['encoder_path'])
vectorizer = pickle.load(io.BytesIO(encoder_bytes['Body'].read()))
X_text = vectorizer.transform(test_texts)
n_missing_features = sklearn_model.coef_.shape[1] - X_text.shape[1]
zeros_features = sp.csr_matrix(np.zeros((X_text.shape[0], n_missing_features)))
X_test = sp.hstack([X_text, zeros_features]).tocsr()

# predictions: 
print(f"Final shape: {X_test.shape}")
sklearn_predictions = sklearn_model.predict(X_test)
test_df['sklearn_predictions'] = sklearn_predictions

print(classification_report(test_df[BIN_COL], test_df['sklearn_predictions']))


Final shape: (1200, 175173)
              precision    recall  f1-score   support

           0       0.73      1.00      0.84       607
           1       1.00      0.63      0.77       593

    accuracy                           0.81      1200
   macro avg       0.86      0.81      0.81      1200
weighted avg       0.86      0.81      0.81      1200



**Выводы**
* F1-мера по классам 0, 1 оптимизирована до 6 и 13 % соответственно
* Recall по токсичному классу вырос почти до 17% 

<!-- {'encode': {'INAPPROPRIATE': 0,
  'INSULT': 1,
  'INSULT,OBSCENITY': 2,
  'INSULT,OBSCENITY,THREAT': 3,
  'INSULT,THREAT': 4,
  'NORMAL': 5,
  'OBSCENITY': 6,
  'OBSCENITY,THREAT': 7,
  'SENSITIVE': 8,
  'THREAT': 9,
  'body_shaming': 10,
  'drugs': 11,
  'gambling': 12,
  'health_shaming': 13,
  'offline_crime': 14,
  'online_crime': 15,
  'politics': 16,
  'pornography': 17,
  'prostitution': 18,
  'racism': 19,
  'religion': 20,
  'sexism': 21,
  'sexual_minorities': 22,
  'slavery': 23,
  'social_injustice': 24,
  'suicide': 25,
  'terrorism': 26,
  'weapons': 27},
 'decode': {0: 'INAPPROPRIATE',
  1: 'INSULT',
  2: 'INSULT,OBSCENITY',
  3: 'INSULT,OBSCENITY,THREAT',
  4: 'INSULT,THREAT',
  5: 'NORMAL',
  6: 'OBSCENITY',
  7: 'OBSCENITY,THREAT',
  8: 'SENSITIVE',
  9: 'THREAT',
  10: 'body_shaming',
  11: 'drugs',
  12: 'gambling',
  13: 'health_shaming',
  14: 'offline_crime',
  15: 'online_crime',
  16: 'politics',
  17: 'pornography',
  18: 'prostitution',
  19: 'racism',
  20: 'religion',
  21: 'sexism',
  22: 'sexual_minorities',
  23: 'slavery',
  24: 'social_injustice',
  25: 'suicide',
  26: 'terrorism',
  27: 'weapons'}} -->